# EXO_04_DARKROOM — Batch Render ATOM-IC

```
╔══════════════════════════════════════════════════════════════════════════════╗
║           PHOTOGRAPHY WING — DARKROOM V1 — BATCH RENDERING                  ║
║                                                                              ║
║   Rendu 1080p @ 128 samples + OIDN (pas 4K direct)                          ║
║   U06 Real-ESRGAN upscale les frames gradées → 4K                           ║
║   Chunks de 300 frames + checkpoint JSON (résiste au timeout 12h)            ║
║   ATOM-IC : Transmutation 1080p → 4K (~2-4h au lieu de 15-45h)              ║
║                                                                              ║
║   Pipeline : scene_ready_*.blend → darkroom_render.py → render_*.png         ║
║   CLI      : EXO_04_DARKROOM.py --drive-root --project-name --resume         ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

**Mode:** Rendu batch headless Blender sur Colab T4. Auto-resume via checkpoint.

## 1. Mount Drive + Configuration

In [ ]:
import os
import sys
import json
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# === CONFIGURATION ===
PROJECT_NAME = "EXODUS_TEST_01"         # <-- Modifier selon le projet
DRIVE_ROOT   = Path("/content/drive/MyDrive/EXODUS_V2_FRESH")  # <-- Modifier si besoin
CHUNK_SIZE   = 300
PRESET       = "preview"               # darkroom (1080p/128) | preview (1080p/64) | production (4K/256)

# === TEST MODE : limiter les frames ===
FRAME_START  = 1                        # 1 = debut
FRAME_END    = 10                       # 10 = test rapide | None = toutes les frames du .blend

UNIT_ROOT  = DRIVE_ROOT / '04_PHOTOGRAPHY_WING'
CODEBASE   = UNIT_ROOT / 'CODEBASE'
OUT_CAMERA = UNIT_ROOT / 'OUT_CAMERA_LOGIC'

sys.path.insert(0, str(CODEBASE))

print(f'Drive Root   : {DRIVE_ROOT}')
print(f'Unit Root    : {UNIT_ROOT}')
print(f'Codebase     : {CODEBASE}')
print(f'OUT_CAMERA   : {OUT_CAMERA}')
print(f'Project      : {PROJECT_NAME}')
print(f'Preset       : {PRESET}')
print(f'Chunk size   : {CHUNK_SIZE}')
print(f'Frames       : {FRAME_START} -> {FRAME_END if FRAME_END else "toutes"}')
print(f'Exists       : {UNIT_ROOT.exists()}')


## 2. Install Blender Headless

In [ ]:
%%bash
# Install Blender 4.0 headless — telecharge depuis blender.org vers /opt/blender-local
BLENDER_BIN=/opt/blender-local/blender
if [ ! -f "$BLENDER_BIN" ]; then
    echo "[DARKROOM] Telechargement Blender 4.0.2..."
    wget -q https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz -O /tmp/blender.tar.xz
    echo "[DARKROOM] Extraction..."
    tar -xf /tmp/blender.tar.xz -C /opt/
    mv /opt/blender-4.0.2-linux-x64 /opt/blender-local
    echo "[DARKROOM] Blender 4.0 installe dans /opt/blender-local"
else
    echo "[DARKROOM] Blender deja present : $BLENDER_BIN"
fi
/opt/blender-local/blender --version | head -1
echo "[DARKROOM] GPU disponible :"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || echo "  Pas de GPU NVIDIA"


## 3. Verify Setup

In [ ]:
import shutil

print("=== Pre-flight Check ===")

# Blender
blender_path = shutil.which("blender")
print(f"  Blender        : {blender_path or 'NOT FOUND'}")

# .blend files
blends = sorted(OUT_CAMERA.glob("scene_ready_*.blend")) if OUT_CAMERA.exists() else []
print(f"  .blend trouvés : {len(blends)}")
for b in blends:
    size_mb = b.stat().st_size / (1024 * 1024)
    print(f"    {b.name} ({size_mb:.1f} MB)")

# Preset
from camera_schema import RENDER_PRESETS
print(f"  Preset '{PRESET}' : {'OK' if PRESET in RENDER_PRESETS else 'MISSING'}")
if PRESET in RENDER_PRESETS:
    p = RENDER_PRESETS[PRESET]
    print(f"    Resolution : {p['resolution']}")
    print(f"    Samples    : {p['samples']}")
    print(f"    Denoiser   : {p['denoiser']}")

# Disk space
stat = shutil.disk_usage("/content")
free_gb = stat.free / (1024 ** 3)
print(f"  Espace disque  : {free_gb:.1f} GB libre")

# Checkpoint
ckpt = OUT_CAMERA / "darkroom_checkpoint.json"
if ckpt.exists():
    with open(ckpt) as f:
        ckpt_data = json.load(f)
    print(f"  Checkpoint     : frame {ckpt_data['next_frame']}/{ckpt_data['total_frames']}")
else:
    print(f"  Checkpoint     : Aucun (démarrage frais)")

# Frames already rendered
existing_frames = sorted(OUT_CAMERA.glob("render_*.png")) if OUT_CAMERA.exists() else []
print(f"  Frames rendues : {len(existing_frames)}")

if not blends:
    print("\nERREUR : Aucun scene_ready_*.blend — lancez d'abord U04-A (EXO_04_PRODUCTION)")
elif blender_path is None:
    print("\nERREUR : Blender non installé — relancez la cellule d'installation")
else:
    print(f"\nOK — Prêt pour le rendu ({len(blends)} scènes)")

## 3.5 VULKAN_FORGE — Injection Caméra + VOID-FLUSH

> **Tech-Prêtre : VULKAN_FORGE** — Fix ID : `VULKAN_CAMERA_FIX_v1`  
> Arsenal : `VULKAN_FORGE/ARSENAL/scripts/inject_camera_cinematic.py`  
> Armes : `WEAPONS/hook_dispatcher.py` + `ADEPTUS/VOID-FLUSH`  
> Rôle : Injecte camera_main (35mm/DOF) + Sun + Point dans chaque scene_ready_*.blend

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# VULKAN_FORGE — CAMERA INJECTION PROTOCOL — VULKAN_CAMERA_FIX_v1
# Insere AVANT le render DARKROOM
# ═══════════════════════════════════════════════════════════════════
import subprocess, shutil, json, sys, os
from pathlib import Path

REPO_VULKAN = Path('/tmp/exodus-vulkan-inject')

# Auto-detection Blender — prefere 4.0 dans /opt, fallback apt
_candidates = ['/opt/blender-local/blender', '/usr/bin/blender', '/usr/local/bin/blender']
BLENDER_BIN = next((Path(p) for p in _candidates if Path(p).exists()),
                   Path(shutil.which('blender') or '/opt/blender-local/blender'))
print(f'[VULKAN] Blender : {BLENDER_BIN} — exists={BLENDER_BIN.exists()}')
if not BLENDER_BIN.exists():
    raise FileNotFoundError('[VULKAN] Blender introuvable — relancez la cellule installation (cellule 4)')

# ── 1. SYNC VULKAN_FORGE DEPUIS GITHUB ──
print('[VULKAN] Sync Arsenal depuis GitHub...')
if REPO_VULKAN.exists():
    shutil.rmtree(REPO_VULKAN)
subprocess.run(['git', 'clone', '--depth', '1',
    'https://github.com/kioka8877-ux/EXODUS-V2.git', str(REPO_VULKAN)],
    capture_output=True, check=True)
INJECT_SCRIPT   = REPO_VULKAN / 'VULKAN_FORGE' / 'ARSENAL' / 'scripts' / 'inject_camera_cinematic.py'
HOOK_DISPATCHER = REPO_VULKAN / 'VULKAN_FORGE' / 'WEAPONS' / 'hook_dispatcher.py'
VOID_FLUSH      = REPO_VULKAN / 'ADEPTUS_EXODUS' / 'magos_physic' / 'VOID-FLUSH' / 'void_flush.py'
print(f'  inject_camera_cinematic.py : {"OK" if INJECT_SCRIPT.exists() else "ABSENT"}')
print(f'  hook_dispatcher.py         : {"OK" if HOOK_DISPATCHER.exists() else "ABSENT"}')
print(f'  void_flush.py              : {"OK" if VOID_FLUSH.exists() else "ABSENT"}')

# ── 1.5 SYNC EXO_04_DARKROOM.py vers Drive (fix --frame-start/--frame-end) ──
_src_darkroom = REPO_VULKAN / "04_PHOTOGRAPHY_WING" / "CODEBASE" / "EXO_04_DARKROOM.py"
_dst_darkroom = CODEBASE / "EXO_04_DARKROOM.py"
if _src_darkroom.exists() and CODEBASE.exists():
    shutil.copy2(str(_src_darkroom), str(_dst_darkroom))
    print("[VULKAN] EXO_04_DARKROOM.py synced depuis GitHub (fix frame-start/frame-end)")
else:
    print(f"[VULKAN] WARN: sync impossible — src={_src_darkroom.exists()} dst_dir={CODEBASE.exists()}")

# ── 2. VOID-FLUSH — Purge VRAM/GC avant render ──
print('\n[VOID-FLUSH] Purge VRAM/GC...')
if VOID_FLUSH.exists():
    r = subprocess.run([sys.executable, str(VOID_FLUSH), '--fregate', 'U04'],
        capture_output=True, text=True, cwd=str(VOID_FLUSH.parent))
    for line in r.stdout.splitlines():
        if '[VOID' in line or 'flush' in line.lower():
            print(f'  {line}')
else:
    import gc; gc.collect()
    print('  GC Python (fallback VOID-FLUSH absent)')

# ── 3. DETECTION BLENDS ──
print('\n[VULKAN] Detection scene_ready_*.blend...')
blends = sorted(OUT_CAMERA.glob('scene_ready_*.blend'))
if not blends:
    u03_out = DRIVE_ROOT / '03_SCENOGRAPHY_DOCK' / 'OUT_PREMIUM_SCENE'
    env_blends = sorted(u03_out.glob('environment_*.blend')) if u03_out.exists() else []
    for i, src in enumerate(env_blends, 1):
        dst = OUT_CAMERA / f'scene_ready_{i:02d}.blend'
        shutil.copy2(str(src), str(dst))
        print(f'  Copie U03 -> {dst.name}')
    blends = sorted(OUT_CAMERA.glob('scene_ready_*.blend'))
print(f'  {len(blends)} blend(s) a traiter')

# ── 4. INJECTION CAMERA DANS CHAQUE BLEND ──
print('\n[VULKAN] Injection camera cinematique...')
inject_results = []
all_ok = True
for blend in blends:
    print(f'  >> {blend.name}')
    r = subprocess.run(
        [str(BLENDER_BIN), '--background', str(blend), '--python', str(INJECT_SCRIPT)],
        capture_output=True, text=True, timeout=120)
    for line in r.stdout.splitlines():
        if '[INJECT_CAMERA]' in line:
            print(f'     {line}')
    if r.returncode != 0:
        print(f'     STDERR: {r.stderr[-300:]}')
        all_ok = False
        inject_results.append({'blend': blend.name, 'status': 'ERROR'})
    else:
        inject_results.append({'blend': blend.name, 'status': 'OK'})

# ── 5. HOOK — fix.applied ──
if HOOK_DISPATCHER.exists():
    payload = json.dumps({'fix_id': 'VULKAN_CAMERA_FIX_v1', 'fregate': 'U04',
        'blends_treated': len(blends), 'all_ok': all_ok})
    r = subprocess.run([sys.executable, str(HOOK_DISPATCHER), 'fix.applied', payload],
        capture_output=True, text=True, cwd=str(REPO_VULKAN / 'VULKAN_FORGE'))
    hook_line = [l for l in r.stdout.splitlines() if '[HOOK]' in l]
    if hook_line:
        print(f'\n{hook_line[-1]}')

# ── BILAN ──
status = 'OK' if all_ok else 'PARTIEL'
print(f'\n[VULKAN] Injection terminee : {len(blends)} blend(s) — statut={status}')
if not all_ok:
    raise RuntimeError('[VULKAN] Injection camera FAILED — verifiez les logs ci-dessus avant de continuer')
print('[VULKAN] Pret pour DARKROOM.')


## 4. Render (avec auto-resume)

In [ ]:
cmd = f'python "{CODEBASE}/EXO_04_DARKROOM.py"'
cmd += f' --drive-root "{DRIVE_ROOT}"'
cmd += f' --project-name "{PROJECT_NAME}"'
cmd += f' --blender-path "/opt/blender-local/blender"'
cmd += f' --chunk-size {CHUNK_SIZE}'
cmd += f' --preset {PRESET}'
if FRAME_END:
    cmd += f' --frame-start {FRAME_START}'
    cmd += f' --frame-end {FRAME_END}'
cmd += ' --resume'
cmd += ' -v'

print(f'=== Commande ===')
print(cmd)
print(f'\n=== Execution ===')
!{cmd}


## 5. Progress Check

In [ ]:
print("=== Progression ===")

# Checkpoint status
ckpt = OUT_CAMERA / "darkroom_checkpoint.json"
if ckpt.exists():
    with open(ckpt) as f:
        data = json.load(f)
    pct = data['frames_rendered'] / data['total_frames'] * 100
    print(f"  Checkpoint : {data['frames_rendered']}/{data['total_frames']} ({pct:.1f}%)")
    print(f"  Next frame : {data['next_frame']}")
    print(f"  Elapsed    : {data['elapsed_seconds'] / 60:.1f} min")
    remaining = data['total_frames'] - data['frames_rendered']
    if data['frames_rendered'] > 0:
        spf = data['elapsed_seconds'] / data['frames_rendered']
        eta = spf * remaining / 60
        print(f"  ETA        : ~{eta:.0f} min ({spf:.1f}s/frame)")
else:
    print("  Pas de checkpoint actif")

# Count frames on disk
frames = sorted(OUT_CAMERA.glob("render_*.png")) if OUT_CAMERA.exists() else []
total_size = sum(f.stat().st_size for f in frames)
print(f"\n  Frames sur disque : {len(frames)}")
print(f"  Taille totale     : {total_size / (1024 * 1024):.1f} MB ({total_size / (1024 ** 3):.2f} GB)")

if frames:
    avg_size = total_size / len(frames)
    print(f"  Taille moyenne    : {avg_size / 1024:.1f} KB/frame")
    print(f"  Première frame    : {frames[0].name}")
    print(f"  Dernière frame    : {frames[-1].name}")

## 6. Verify Output

In [ ]:
print("=== Vérification Output ===")

frames = sorted(OUT_CAMERA.glob("render_*.png")) if OUT_CAMERA.exists() else []
total_size = sum(f.stat().st_size for f in frames)

print(f"  Frames PNG     : {len(frames)}")
print(f"  Taille totale  : {total_size / (1024 ** 3):.2f} GB")

# Check for gaps
if frames:
    numbers = [int(f.stem.split('_')[1]) for f in frames]
    expected = set(range(min(numbers), max(numbers) + 1))
    actual = set(numbers)
    missing = expected - actual
    if missing:
        print(f"  ATTENTION : {len(missing)} frames manquantes")
        print(f"    Exemples : {sorted(missing)[:10]}")
    else:
        print(f"  Séquence   : continue ({min(numbers)}–{max(numbers)})")

# Report
report = OUT_CAMERA / "darkroom_report.json"
if report.exists():
    with open(report) as f:
        data = json.load(f)
    s = data.get('summary', {})
    print(f"\n  Rapport :")
    print(f"    Scènes         : {s.get('total_scenes', 'N/A')}")
    print(f"    Frames rendues : {s.get('total_frames', 'N/A')}")
    print(f"    Temps total    : {s.get('total_elapsed_seconds', 0) / 60:.1f} min")
    print(f"    Moyenne        : {s.get('avg_seconds_per_frame', 'N/A')}s/frame")
else:
    print(f"\n  Rapport : Non trouvé")

# Checkpoint cleanup
ckpt = OUT_CAMERA / "darkroom_checkpoint.json"
if ckpt.exists():
    print(f"\n  ATTENTION : Checkpoint encore présent — rendu peut-être incomplet")
    print(f"  Relancez la cellule Render pour reprendre")
else:
    print(f"\n  Checkpoint : supprimé (rendu complet)")

print("\n" + "=" * 60)
print("   DARKROOM — VÉRIFICATION TERMINÉE")
print("=" * 60)